In [5]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.remote("sc://spark-connect.spark.svc.cluster.local:15002").getOrCreate()
print(spark.version)

4.1.2


In [21]:
from pyspark.sql import functions as F

data = [
    ("alice", "eng", 100),
    ("bob",   "eng", 120),
    ("carol", "sales", 90),
    ("dave",  "sales", 110),
]
df = spark.createDataFrame(data, ["name", "dept", "score"])

(df.groupBy("dept")
   .agg(F.avg("score").alias("avg_score"),
        F.count("*").alias("headcount"))
   .orderBy("dept")
   .show())

+-----+---------+---------+
| dept|avg_score|headcount|
+-----+---------+---------+
|  eng|    110.0|        2|
|sales|    100.0|        2|
+-----+---------+---------+



In [ ]:
from pyspark.sql import functions as F
import time

start = time.time()

try:
    # 100 million rows
    df = (
        spark.range(0, 100_000_000)
        .withColumn("group", F.col("id") % 1000)
        .withColumn("value", F.rand() * 1000)
        .repartition(64, "group")
    )

    result = (
        df.groupBy("group")
        .agg(
            F.count("*").alias("cnt"),
            F.avg("value").alias("avg_value"),
            F.stddev("value").alias("stddev_value"),
            F.min("value").alias("min_value"),
            F.max("value").alias("max_value"),
        )
        .orderBy("group")
        .cache()
    )

    result.show(20)
    result.count()  # materialize cache

finally:
    end = time.time()
    print(f"estimated time: {end - start:.3f} seconds")

+-----+------+------------------+------------------+--------------------+-----------------+
|group|   cnt|         avg_value|      stddev_value|           min_value|        max_value|
+-----+------+------------------+------------------+--------------------+-----------------+
|    0|100000| 500.1416387351752| 289.0649010892294| 0.01820017894227366|999.9998436452346|
|    1|100000|499.91226123550535| 288.2253903165856|0.003345640888330337| 999.990501152949|
|    2|100000|500.44584990213156|288.31122841429266|0.001796025575817...|999.9971701393564|
|    3|100000|501.10513747895607| 288.6569577033711|0.002692526126724104|  999.98549531424|
|    4|100000| 499.7975652080614| 288.3437649413526|0.001296135961004...|999.9988333947857|
|    5|100000| 500.1294144599993|288.95980351949527|0.004000502400014483|999.9833174760199|
|    6|100000|  500.681395699901| 289.5249098180177|0.002529076278623421|999.9995521578751|
|    7|100000|  498.030298612073|289.00962126765666|5.668053533192108E-4|999.997